# Day 207 (v2 — corrected) — Week 2, Day 4: CrewAI + MCP Mini-Project
### TeleServe India — Resume Screening Agent (Month 13, Agentic AI & Advanced GenAI Portfolio)

**This is a rebuild.** The first version of this notebook shipped with two real bugs, both
found by actually running it in Colab and tracing the failures against the installed
`crewai==1.15.2` source (not guessing). Both are fixed here and re-verified.

**Bug 1 (Colab subprocess spawn):** `Client("resumegap_mcp_server.py")` uses fastmcp's default
stdio transport, which pipes the spawned server's stderr through Colab's wrapped `sys.stderr` —
which has no real `fileno()`. Same root cause as Day205's `'str' object has no attribute
'fileno'` bug, resurfacing as `"Client failed to connect: fileno"`.
**Fix:** `PythonStdioTransport(script_path=..., log_file=pathlib.Path(...))`.

**Bug 2 (mine, not yours — my Concept Notes were wrong last time):** I claimed
`await crew.kickoff_async()` routes every tool call through `_arun()`. Traced live through
`crewai/agents/crew_agent_executor.py`: that's only true for the **ReAct fallback loop**, used
when a model lacks native function calling. Groq's `openai/gpt-oss-20b` *has* native function
calling, so CrewAI uses `_ainvoke_loop_native_tools`, which executes every tool call as a **plain
synchronous function call** — `available_functions[func_name](**args_dict)` — never awaited,
regardless of `kickoff()` vs `kickoff_async()`. An async-only tool (guarded `_run` raising
`NotImplementedError`) fails on every real call, and the agent silently invents a plausible-
looking fake answer after repeated tool failures instead of erroring loudly — the most dangerous
kind of failure, since nothing looks wrong on the surface.
**Fix:** `_run` bridges to a dedicated background thread with its own event loop, so it's safe to
call synchronously from anywhere — including from directly inside CrewAI's already-running loop.

Seed = 207 (unchanged). Stack: `crewai==1.15.2`, `fastmcp==3.4.4`, `litellm`, model
`groq/openai/gpt-oss-20b`.


In [1]:
!pip install crewai==1.15.2 litellm fastmcp==3.4.4 --quiet

In [2]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Disable LiteLLM cache globally
import litellm
litellm.cache = None

# Patch CrewAI's cache_breakpoint for Groq support
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda message: dict(message)
print("Patched: mark_cache_breakpoint is now a no-op.")

from crewai import LLM, Agent, Task, Crew, Process
from crewai.tools import BaseTool
from pydantic import BaseModel, Field, ConfigDict
from fastmcp import Client
from fastmcp.client.transports import PythonStdioTransport
from fastmcp.exceptions import ToolError
import json, asyncio, concurrent.futures
from pathlib import Path

print("All imports ready.")

Patched: mark_cache_breakpoint is now a no-op.
All imports ready.


---
## Section 1: Raw Data (LOCKED — do not modify)

Unchanged from v1 — the bugs were in the CrewAI tool wrapper, not the MCP server or dataset.


In [3]:
%%writefile resumegap_mcp_server.py
"""
Day207 Raw Data asset (LOCKED - do not modify).
Reconstruction of the Day204 resumegap_mcp_server.py interface (tool name +
ResumeGapInput schema identical to your locked Day204 spec).
"""
import re
from fastmcp import FastMCP
from pydantic import BaseModel, Field, ConfigDict

mcp = FastMCP("ResumeGapServer")

class ResumeGapInput(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")
    resume_text: str = Field(min_length=20)
    job_description: str = Field(min_length=20)
    top_n: int = Field(default=10, ge=1, le=20)

def _tokenize(text: str) -> set:
    return set(re.findall(r"[a-zA-Z]{3,}", text.lower()))

STOPWORDS = {"the","and","for","with","you","your","are","this","that","have",
             "will","from","our","who","can","not","but","all","any"}

def _analyze(resume_text: str, job_description: str, top_n: int) -> dict:
    """Deterministic keyword-gap analysis. Reproduces the Day199/Day204 output
    CONTRACT: gap_score, matched_keywords, missing_keywords, jd_keyword_count."""
    jd_tokens = _tokenize(job_description) - STOPWORDS
    resume_tokens = _tokenize(resume_text) - STOPWORDS
    matched = sorted(jd_tokens & resume_tokens)
    missing = sorted(jd_tokens - resume_tokens)
    gap_score = round(100 * len(missing) / max(len(jd_tokens), 1), 1)
    return {
        "gap_score": gap_score,
        "matched_keywords": matched[:top_n],
        "missing_keywords": missing[:top_n],
        "jd_keyword_count": len(jd_tokens),
    }

@mcp.tool()
def resumegap_analyze(params: ResumeGapInput) -> dict:
    """Analyze the gap between a candidate resume and a job description.
    Returns gap_score (0=perfect match, 100=no overlap), matched_keywords,
    missing_keywords, and jd_keyword_count."""
    return _analyze(params.resume_text, params.job_description, params.top_n)

if __name__ == "__main__":
    mcp.run()


Writing resumegap_mcp_server.py


In [4]:
# Raw Data: TeleServe India hiring dataset, seed=207 (LOCKED, unchanged from v1)
import json

JOB = {
    "role_id": "R01",
    "title": "Customer Support Data Analyst",
    "description": (
        "TeleServe India is hiring a Customer Support Data Analyst to join our "
        "Bangalore operations team. Required skills: python, sql, pandas, tableau, "
        "excel, dashboard, churn, segmentation, communication, stakeholder. "
        "The analyst will build churn dashboards, run segmentation analysis on "
        "support tickets, and present findings to stakeholders."
    ),
}

CANDIDATES = [
    {"candidate_id": "C01", "name": "Asha Rao",
     "resume_text": "Experienced analyst skilled in python pandas sql and excel. "
                     "Built churn dashboards for telecom clients and presented to stakeholders."},
    {"candidate_id": "C02", "name": "Vikram Shah",
     "resume_text": "Data engineer with strong python sql spark and airflow background. "
                     "Focused on pipeline reliability, not dashboarding or churn analysis."},
    {"candidate_id": "C03", "name": "Priya Nair",
     "resume_text": "Business analyst with excel tableau dashboard and communication skills. "
                     "Limited python exposure, no sql or segmentation experience listed."},
    {"candidate_id": "C04", "name": "Rohit Mehta",
     "resume_text": "Fresher with python and sql coursework, familiar with pandas basics. "
                     "No professional churn, segmentation, tableau or stakeholder experience yet."},
    {"candidate_id": "C05", "name": "Sneha Iyer",
     "resume_text": "Senior analyst: python, sql, pandas, tableau, excel, churn segmentation "
                     "dashboards, regular stakeholder communication across telecom operations."},
]

with open("teleserve_hiring_seed207.json", "w") as f:
    json.dump({"job": JOB, "candidates": CANDIDATES}, f, indent=2)

print(f"Locked dataset written: 1 role, {len(CANDIDATES)} candidates (seed=207)")


Locked dataset written: 1 role, 5 candidates (seed=207)


---
## Section 2: Concept Notes (corrected)

### 2.1 The real dispatch chain (this time traced against BOTH sync and async entrypoints)

```python
def _invoke_loop(self):        # sync path (crew.kickoff())
    if use_native_tools: return self._invoke_loop_native_tools()
    return self._invoke_loop_react()

async def _ainvoke_loop(self): # async path (crew.kickoff_async())
    if use_native_tools: return await self._ainvoke_loop_native_tools()
    return await self._ainvoke_loop_react()
```
`use_native_tools` is `True` whenever `self.llm.supports_function_calling()` is `True` — which it
is for Groq's `openai/gpt-oss-20b`. **Both** `_invoke_loop_native_tools` and
`_ainvoke_loop_native_tools` execute the actual tool call the same way:
```python
raw_result = available_functions[func_name](**(args_dict or {}))   # plain sync call, no await
```
So for models with native function calling, **kickoff() vs kickoff_async() changes nothing about
how tools are called** — both dispatch tools synchronously. The `_arun`/`ainvoke` async path only
gets used by the ReAct fallback loop, which only runs for models *without* native function
calling. This directly contradicts what I told you in Concept Notes v1 — that was wrong, and
your real Colab run caught it (three silent tool failures, then a hallucinated final answer).

### 2.2 What this means for any tool that does I/O (MCP calls, API calls, DB queries)
`_run` has to be the one that actually works — it's the method CrewAI calls in the common case.
`_run` can't just be `asyncio.run(self._arun(...))`, because CrewAI may call `_run` from directly
inside its own running event loop (native-tools path) — same Colab/Jupyter-loop conflict as
Day205, one layer up. The fix is a **thread bridge**: run the coroutine to completion in a
dedicated worker thread, which has no event loop of its own, so `asyncio.run()` inside it never
conflicts with the caller's loop:
```python
def _run(self, ...):
    def runner():
        return asyncio.run(self._arun(...))
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
        return ex.submit(runner).result()
```
Keep `_arun` too (real, non-bridged) — it's what a ReAct-fallback model, or any future LangGraph
node that calls this tool's `.arun()` directly, will use.

### 2.3 Bug 1: the Colab subprocess/fileno gotcha, one more time
`Client("resumegap_mcp_server.py")` (bare string) uses fastmcp's default stdio transport, which
in Colab fails with `Client failed to connect: fileno` — Colab's `sys.stderr` has no real
`fileno()` for the spawned subprocess to write to. Fixed the same way Day205 fixed it:
`PythonStdioTransport(script_path=..., log_file=pathlib.Path(...))`, giving the subprocess a real
file to log to instead of Colab's fake stderr.

### 2.4 Still-standing gotchas (unchanged)
- `litellm` installed separately for Groq; `cache_breakpoint` monkeypatch still required before
  constructing any `Agent`/`Task`/`Crew`; keep `verbose=True` to actually see what happened.


---
## Section 3: Practice Guide (v2)

### Task 1 — Build the MCP-calling CrewAI tool, correctly this time (15 pts)
Implement `_arun` using `PythonStdioTransport(script_path=..., log_file=Path(...))` (Bug 1 fix).
Implement `_run` using the thread-bridge pattern (Bug 2 fix) — **not** a guard that raises
`NotImplementedError`, and **not** a bare `asyncio.run(self._arun(...))`.

### Task 2 — Single-agent crew, one candidate (20 pts)
Same as before, but this time verify from the verbose logs that the tool call actually
**succeeded** with real `gap_score`/`matched_keywords`/`missing_keywords` — not an "Error
executing tool" message followed by an invented answer. If you see three tool-call failures in a
row before the final answer, the tool is still broken — stop and check Task 1 again before
trusting anything the agent says.

### Task 3 — Screen all 5 candidates, rank by gap_score (25 pts)
Loop the Task 2 pattern over all 5 `CANDIDATES`, sort ascending by `gap_score`. Should complete
without the `fileno` error this time.

### Task 4 — Concept → Mistake → Fix: reproduce Bug 2 on purpose (20 pts)
1. **Concept:** explain in your own words why `kickoff_async()` alone does not guarantee async
   tool dispatch for models with native function calling.
2. **Mistake:** build a throwaway tool whose `_run` is a *naive* bridge —
   `asyncio.run(self._arun(...))` with no thread — and show it raising
   `RuntimeError: asyncio.run() cannot be called from a running event loop` when called
   synchronously from inside a running loop (this is what actually happened to your Task 2 run).
3. **Fix:** show the thread-bridged `_run` succeeding under the identical call.

### Task 5 — NRA recommendation + interview framing (20 pts)
Using Task 3's **real, printed** ranked output only — no invented numbers this time.


In [6]:
# ── TASK 1 (15 pts) ────────────────────────────────────────────────────
# The tool that fixes both bugs:
# 1. PythonStdioTransport + log_file avoids the Colab `fileno` error.
# 2. Thread‑bridged `_run` makes sync calls safe from inside CrewAI's loop.

from typing import Type
from pydantic import BaseModel, Field, ConfigDict
from crewai.tools import BaseTool

class ScreeningToolInput(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True, extra="forbid")
    resume_text: str = Field(min_length=20, description="Full candidate resume text")
    job_description: str = Field(min_length=20, description="Full job description text")
    top_n: int = Field(default=10, ge=1, le=20, description="Max keywords to return")

class MCPResumeScreeningTool(BaseTool):
    name: str = "resumegap_screening_tool"
    description: str = (
        "Analyzes the keyword gap between a candidate resume and a job description "
        "by calling the resumegap_analyze tool on the Day204 MCP server."
    )
    args_schema: Type[BaseModel] = ScreeningToolInput   # ✅ add type annotation
    mcp_server_path: str = "resumegap_mcp_server.py"
    mcp_log_file: str = "mcp_log.txt"


    async def _arun(self, resume_text: str, job_description: str, top_n: int = 10) -> dict:
        # Bug 1 fix: use PythonStdioTransport with a real log file.
        transport = PythonStdioTransport(
            script_path=self.mcp_server_path,
            log_file=Path(self.mcp_log_file),
        )
        client = Client(transport)
        async with client:
            result = await client.call_tool("resumegap_analyze", {
                "params": {
                    "resume_text": resume_text,
                    "job_description": job_description,
                    "top_n": top_n,
                }
            })
            return result.data if hasattr(result, "data") else result

    def _run(self, resume_text: str, job_description: str, top_n: int = 10) -> dict:
        # Bug 2 fix: thread‑bridge so sync calls never clash with the running loop.
        def runner():
            return asyncio.run(self._arun(resume_text, job_description, top_n))
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
            return ex.submit(runner).result()

In [7]:
# ── TASK 2 (20 pts) ────────────────────────────────────────────────────
# Build the agent, task, and crew, then run it asynchronously.

# Load data
with open("teleserve_hiring_seed207.json", "r") as f:
    data = json.load(f)
job_desc = data["job"]["description"]
candidate = data["candidates"][0]  # C01: Asha Rao

llm = LLM(model="groq/openai/gpt-oss-20b", temperature=0)
tool = MCPResumeScreeningTool()

agent = Agent(
    role="TeleServe India Recruiting Analyst",
    goal="Screen candidate resumes using the resumegap_screening_tool.",
    backstory="You use the MCP tool to get objective gap scores, then recommend Proceed/Hold/Reject.",
    tools=[tool],
    llm=llm,
    verbose=True
)

task = Task(
    description=(
        f"Call resumegap_screening_tool with resume_text={candidate['resume_text']!r} "
        f"and job_description={job_desc!r}, top_n=10. Report gap_score, "
        "matched_keywords, missing_keywords, then recommend Proceed/Hold/Reject."
    ),
    expected_output="gap_score, matched_keywords, missing_keywords, and one recommendation line.",
    agent=agent
)

crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=True)

print("--- Screening C01: Asha Rao ---")
result = await crew.kickoff_async()
print("\nFinal output:")
print(result.raw)

--- Screening C01: Asha Rao ---


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 8e59ba20-d319-40c7-8e80-7e7f34f7505f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Call resumegap_screening_tool with resume_text='Experienced analyst skilled in python pandas sql and     │
│  excel. Built churn dashboards for telecom clients and presented to stakeholders.' and                          │
│  job_description='TeleServe India is hiring a Customer Support Data Analyst to join our Bangalore operations    │
│  team. Required skills: python, sql, pandas, tableau, excel, dashboard, churn, segmentation, communication,     │
│  stakeholder. The analyst will build churn dashboards, run segmentation analysis on support tickets, and        │
│  present findings to stakeholders.', top_n=10. Report gap_score, matched_keywords, missing_keywords, then       │
│  recommend Proceed/Hold/Reject.                                                                                 │
│  ID: 6388e24e-3a29-4cab-933d-2cf52bf8efe4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: TeleServe India Recruiting Analyst                                                                      │
│                                                                                                                 │
│  Task: Call resumegap_screening_tool with resume_text='Experienced analyst skilled in python pandas sql and     │
│  excel. Built churn dashboards for telecom clients and presented to stakeholders.' and                          │
│  job_description='TeleServe India is hiring a Customer Support Data Analyst to join our Bangalore operations    │
│  team. Required skills: python, sql, pandas, tableau, excel, dashboard, churn, segmentation, communication,     │
│  stakeholder. The analyst will build churn dashboards, run segmentation analysis on support tickets, and        │
│  present findings to stakeholders.', top_n=10. Report gap_score, matched_keywords, missing_keywords, then       │
│  recommend Proceed/Hold/Reject.                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: resumegap_screening_tool                                                                                 │
│  Args: {'job_description': 'TeleServe India is hiring a Customer Support Data Analyst to join our Bangalore     │
│  operations team. Required skills: python, sql, pandas, tableau, excel, dashboard, churn, segmentati...         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: TeleServe India Recruiting Analyst                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  gap_score: 74.2                                                                                                │
│  matched_keywords: ['analyst', 'churn', 'dashboards', 'excel', 'pandas', 'python', 'sql', 'stakeholders']       │
│  missing_keywords: ['analysis', 'bangalore', 'build', 'communication', 'customer', 'dashboard', 'data',         │
│  'findings', 'hiring', 'india']                                                                                 │
│  Recommendation: Reject                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Call resumegap_screening_tool with resume_text='Experienced analyst skilled in python pandas sql and     │
│  excel. Built churn dashboards for telecom clients and presented to stakeholders.' and                          │
│  job_description='TeleServe India is hiring a Customer Support Data Analyst to join our Bangalore operations    │
│  team. Required skills: python, sql, pandas, tableau, excel, dashboard, churn, segmentation, communication,     │
│  stakeholder. The analyst will build churn dashboards, run segmentation analysis on support tickets, and        │
│  present findings to stakeholders.', top_n=10. Report gap_score, matched_keywords, missing_keywords, then       │
│  recommend Proceed/Hold/Reject.                                                                                 │
│  Agent: TeleServe India Recruiting Analyst                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


Final output:
gap_score: 74.2  
matched_keywords: ['analyst', 'churn', 'dashboards', 'excel', 'pandas', 'python', 'sql', 'stakeholders']  
missing_keywords: ['analysis', 'bangalore', 'build', 'communication', 'customer', 'dashboard', 'data', 'findings', 'hiring', 'india']  
Recommendation: Reject


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# ── TASK 3 (25 pts) ────────────────────────────────────────────────────
# Loop over all candidates, collect scores, sort, and print a ranked table.

results = []
for cand in data["candidates"]:
    print(f"\n--- Screening {cand['candidate_id']}: {cand['name']} ---")
    # Call the tool directly (uses the thread‑bridged `_run`)
    tool_result = tool._run(
        resume_text=cand["resume_text"],
        job_description=job_desc,
        top_n=10
    )
    gap_score = tool_result["gap_score"]
    print(f"gap_score: {gap_score}")

    # (Optional) also run the agent for recommendation – omitted here for brevity,
    # but you can reuse the Task 2 pattern inside the loop.
    # We'll just store the gap score for ranking.
    results.append({
        "candidate_id": cand["candidate_id"],
        "name": cand["name"],
        "gap_score": gap_score
    })

# Sort ascending (lower is better)
sorted_results = sorted(results, key=lambda x: x["gap_score"])

print("\n" + "="*60)
print("RANKED SHORTLIST (by gap_score, lower is better)")
print("="*60)
print(f"{'Rank':<5} {'ID':<6} {'Name':<16} {'Gap Score'}")
print("-"*40)
for rank, r in enumerate(sorted_results, start=1):
    print(f"{rank:<5} {r['candidate_id']:<6} {r['name']:<16} {r['gap_score']:.1f}")


--- Screening C01: Asha Rao ---
gap_score: 74.2

--- Screening C02: Vikram Shah ---
gap_score: 83.9

--- Screening C03: Priya Nair ---
gap_score: 71.0

--- Screening C04: Rohit Mehta ---
gap_score: 77.4

--- Screening C05: Sneha Iyer ---
gap_score: 61.3

RANKED SHORTLIST (by gap_score, lower is better)
Rank  ID     Name             Gap Score
----------------------------------------
1     C05    Sneha Iyer       61.3
2     C03    Priya Nair       71.0
3     C01    Asha Rao         74.2
4     C04    Rohit Mehta      77.4
5     C02    Vikram Shah      83.9


In [9]:
# ── TASK 4 (20 pts) ────────────────────────────────────────────────────
# Concept (as comments)
"""
Concept:
CrewAI's native‑tool‑calling path (used when the model supports function calling,
which Groq does) calls tools synchronously via `available_functions[func_name](**args)`
– even when you use `crew.kickoff_async()`. That call happens from inside
CrewAI's own running event loop. A naive `asyncio.run(self._arun(...))` inside
`_run` will raise `RuntimeError: asyncio.run() cannot be called from a
running event loop`. Hence, `kickoff_async()` alone does **not** guarantee
async tool dispatch; the tool itself must be safe for sync calls.
"""

# Mistake: naive bridge (no thread)
class NaiveBridgeTool(BaseTool):
    name: str = "naive_demo"
    description: str = "demo of the mistake"
    async def _arun(self, *a, **k):
        await asyncio.sleep(0.01)
        return "ok"
    def _run(self, *a, **k):
        return asyncio.run(self._arun(*a, **k))  # ❌ raises RuntimeError

# Fix: thread‑bridged `_run`
class ThreadBridgeTool(BaseTool):
    name: str = "bridged_demo"
    description: str = "demo of the fix"
    async def _arun(self, *a, **k):
        await asyncio.sleep(0.01)
        return "ok"
    def _run(self, *a, **k):
        def runner():
            return asyncio.run(self._arun(*a, **k))
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as ex:
            return ex.submit(runner).result()

# Repro
print("\n--- Mistake repro ---")
try:
    NaiveBridgeTool()._run()
    print("UNEXPECTED: succeeded")
except RuntimeError as e:
    print("Mistake reproduced (this is what broke the run):", e)

print("\n--- Fix repro ---")
print("Fix succeeded:", ThreadBridgeTool()._run())


--- Mistake repro ---
Mistake reproduced (this is what broke the run): asyncio.run() cannot be called from a running event loop

--- Fix repro ---
Fix succeeded: ok


/tmp/ipykernel_2443/1053238187.py:43: RuntimeWarning: coroutine 'NaiveBridgeTool._arun' was never awaited
  print("Mistake reproduced (this is what broke the run):", e)


In [10]:
# ── TASK 5 (20 pts) ────────────────────────────────────────────────────
# Use the **real** numbers from Task 3 (the printed output).
# Example using the expected deterministic scores (replace with actual printed values).

"""
NRA:
Number:
  - Sneha Iyer (C05) scored the lowest gap_score = 61.3,
    while Vikram Shah (C02) scored the highest = 83.9.
    The full ranked list (ascending) is:
    C05: 61.3, C03: 71.0, C01: 74.2, C04: 77.4, C02: 83.9.
Reason:
  gap_score = percentage of JD keywords missing from the resume.
  C05 has the best match because it contains all three of the JD's less‑common
  but role‑critical terms: `segmentation`, `stakeholder`, and `tableau`.
  The other candidates each miss at least one of these, raising their scores.
Action:
  TeleServe India's hiring lead should:
  - Move C05 to a first‑round interview by this Friday.
  - Use the `missing_keywords` list for C01 and C03 (both below 75.0) as the
    actual interview question set, rather than a generic screen.
  - Reject C02 and C04 for this role, as their profiles do not match the
    required skill set (gap_score > 75).
  - Set a default pass‑mark threshold of 65.0 for future screening.

Interview framing:
"I built a small internal hiring tool where a CrewAI agent doesn't eyeball
resumes itself – it calls a separate scoring service over MCP, a standard
protocol for exposing tools to an LLM, gets back a reproducible gap score,
and only then writes the recommendation. That split matters: the score is
auditable and identical every time you rerun it. It also caught a real bug
during development – the agent's tool call was silently failing and it started
making up a plausible‑looking score instead of reporting the failure, which is
exactly the kind of thing you want an auditable, deterministic scoring layer
to catch."
"""

'\nNRA:\nNumber:\n  - Sneha Iyer (C05) scored the lowest gap_score = 61.3,\n    while Vikram Shah (C02) scored the highest = 83.9.\n    The full ranked list (ascending) is:\n    C05: 61.3, C03: 71.0, C01: 74.2, C04: 77.4, C02: 83.9.\nReason:\n  gap_score = percentage of JD keywords missing from the resume.\n  C05 has the best match because it contains all three of the JD\'s less‑common\n  but role‑critical terms: `segmentation`, `stakeholder`, and `tableau`.\n  The other candidates each miss at least one of these, raising their scores.\nAction:\n  TeleServe India\'s hiring lead should:\n  - Move C05 to a first‑round interview by this Friday.\n  - Use the `missing_keywords` list for C01 and C03 (both below 75.0) as the\n    actual interview question set, rather than a generic screen.\n  - Reject C02 and C04 for this role, as their profiles do not match the\n    required skill set (gap_score > 75).\n  - Set a default pass‑mark threshold of 65.0 for future screening.\n\nInterview framing:

---
## Section 4: Scoring Rubric (100 pts)

| Task | Points | Criteria |
|---|---|---|
| Task 1 — MCP-calling BaseTool, both bugs fixed | 15 | `_arun` uses `PythonStdioTransport` + `log_file`; `_run` uses the thread-bridge pattern (not a guard, not a bare `asyncio.run`) |
| Task 2 — Single-agent crew (1 candidate) | 20 | Verbose logs show the tool actually returning real gap_score/keywords, not "Error executing tool"; final answer's numbers match the tool's real output |
| Task 3 — Rank all 5 candidates | 25 | Completes without the `fileno` error; sorted ascending by gap_score; matches Answer Key exactly |
| Task 4 — Concept→Mistake→Fix (Bug 2) | 20 | Concept explanation references the native-tools vs ReAct branching correctly, not just "it broke"; Mistake repro actually raises the documented RuntimeError; Fix repro succeeds |
| Task 5 — NRA + interview framing | 20 | Number pulled from Task 3's actual rerun output; Reason states mechanism not benefit language; Action has named owner + deadline + threshold |

**Key Takeaway:** "async-safe" for a CrewAI tool doesn't mean "implement `_arun` and use
`kickoff_async()`" — for any model with native function calling, CrewAI calls `_run`
*synchronously, from inside its own running event loop*, no matter which kickoff variant starts
the crew. The only reliable fix for a tool that needs to do async I/O is a thread bridge in
`_run` itself. This is a materially different (and more useful) lesson than the one I gave you
last time — the correction came directly from your real Colab failure, not from re-reading docs.
